In [ ]:

!pip -q install openai playwright pandas
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 19.9 MB/s eta 0:00:00
186.8 MiB [] 0% 0.0s186.8 MiB [] 0% 30.3s186.8 MiB [] 0% 16.3s186.8 MiB [] 0% 16.5s186.8 MiB [] 0% 9.2s186.8 MiB [] 1% 5.2s186.8 MiB [] 2% 3.9s186.8 MiB [] 3% 3.3s186.8 MiB [] 4% 3.1s186.8 MiB [] 4% 2.9s186.8 MiB [] 5% 2.7s186.8 MiB [] 6% 2.6s186.8 MiB [] 6% 2.8s186.8 MiB [] 7% 2.9s186.8 MiB [] 7% 3.1s186.8 MiB [] 7% 3.3s186.8 MiB [] 7% 3.4s186.8 MiB [] 8% 3.2s186.8 MiB [] 9% 3.0s186.8 MiB [] 10% 2.8s186.8 MiB [] 11% 2.6s186.8 MiB [] 12% 2.5s186.8 MiB [] 13% 2.4s186.8 MiB [] 14% 2.4s186.8 MiB [] 14% 2.3s186.8 MiB [] 15% 2.3s186.8 MiB [] 16% 2.2s186.8 MiB [] 17% 2.2s186.8 MiB [] 17% 2.3s186.8 MiB [] 18% 2.2s186.8 MiB [] 19% 2.3s186.8 MiB [] 20% 2.3s186.8 MiB [] 20% 2.4s186.8 MiB [] 21% 2.3s186.8 MiB [] 22% 2.3s186.8 MiB [] 23% 2.2s186.8 MiB [] 24% 2.1s186.8 MiB [] 24% 2.2s186.8 MiB [] 25% 2.3s186.8 MiB [] 25% 2.2s186.8 MiB [] 26%

In [ ]:

# ============================================================
# TESTFORGE AI - AI SOFTWARE TESTING AGENT
# Google Colab + OpenRouter + Playwright
# ============================================================

import os
import re
import json
import asyncio
import pandas as pd

from datetime import datetime
from getpass import getpass
from IPython.display import display, HTML, FileLink

from openai import OpenAI
from playwright.async_api import async_playwright


# ============================================================
# 1. CONFIGURATION
# ============================================================

print("=" * 60)
print("       TESTFORGE AI - SOFTWARE TESTING AGENT")
print("=" * 60)

api_key = getpass("Enter your OpenRouter API key: ")

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

MODEL = "openai/gpt-4o-mini"

print("\nTestForge AI initialized successfully!")


# ============================================================
# 2. GET USER INPUT
# ============================================================

target_url = input(
    "\nEnter the website URL you are authorized to test: "
).strip()

if not target_url.startswith(("http://", "https://")):
    target_url = "https://" + target_url

requirements = input(
    "\nDescribe what you want to test: "
).strip()

number_of_tests = input(
    "\nNumber of test cases (default 8): "
).strip()

number_of_tests = int(number_of_tests or "8")
number_of_tests = max(1, min(number_of_tests, 15))

print("\nTarget website:", target_url)
print("Testing requirements:", requirements)


# ============================================================
# 3. OPENROUTER LLM HELPER
# ============================================================

def ask_ai(prompt, temperature=0.2):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are TestForge AI, a software testing "
                    "expert. Be accurate, concise, and never "
                    "invent browser observations."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature
    )

    return response.choices[0].message.content


# ============================================================
# 4. WEBSITE ANALYSIS AGENT
# ============================================================

async def inspect_website(url):

    print("\n[Agent 1] Inspecting the website...")

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox"]
        )

        page = await browser.new_page()

        try:

            response = await page.goto(
                url,
                wait_until="domcontentloaded",
                timeout=45000
            )

            await page.wait_for_timeout(1500)

            title = await page.title()

            final_url = page.url

            page_text = await page.locator("body").inner_text(
                timeout=10000
            )

            # Limit the amount of text sent to the LLM.
            page_text = page_text[:7000]

            buttons = await page.locator("button").all_inner_texts()

            links = await page.locator("a").evaluate_all(
                """elements => elements.slice(0, 30).map(a => ({
                    text: (a.innerText || a.getAttribute('aria-label') || '').trim(),
                    href: a.href
                }))"""
            )

            inputs = await page.locator("input").evaluate_all(
                """elements => elements.slice(0, 25).map(e => ({
                    type: e.type,
                    name: e.name,
                    placeholder: e.placeholder,
                    id: e.id,
                    ariaLabel: e.getAttribute('aria-label')
                }))"""
            )

            page_info = {
                "requested_url": url,
                "final_url": final_url,
                "title": title,
                "http_status": response.status if response else None,
                "page_text": page_text,
                "buttons": buttons[:30],
                "links": links,
                "inputs": inputs
            }

            await browser.close()

            print("[Agent 1] Website inspection completed.")

            return page_info

        except Exception as e:

            await browser.close()

            raise RuntimeError(
                f"Website inspection failed: {e}"
            )


# ============================================================
# 5. TEST CASE GENERATION AGENT
# ============================================================

def generate_test_cases(page_info, requirements, count):

    print("\n[Agent 2] Generating test cases using AI...")

    prompt = f"""
You are an expert software test engineer.

Generate {count} useful, non-destructive browser test cases.

WEBSITE INFORMATION:
{json.dumps(page_info, indent=2)[:10000]}

USER REQUIREMENTS:
{requirements}

Return ONLY valid JSON in this format:

{{
  "test_cases": [
    {{
      "name": "Page title is correct",
      "description": "Verify the page title",
      "checks": [
        {{
          "type": "title_contains",
          "expected": "some title text"
        }}
      ]
    }}
  ]
}}

SUPPORTED CHECK TYPES:

1. page_load
   {{"type": "page_load"}}

2. title_contains
   {{"type": "title_contains", "expected": "text"}}

3. url_contains
   {{"type": "url_contains", "expected": "text"}}

4. text_visible
   {{"type": "text_visible", "expected": "visible text"}}

5. element_exists
   {{"type": "element_exists", "selector": "CSS selector"}}

RULES:

- Use only the supported check types.
- Use actual website information where possible.
- Use exact text or selectors supported by the observed page.
- Do not invent buttons, forms, or page content.
- Do not generate tests requiring login credentials.
- Do not submit forms or modify data.
- Do not perform purchases, deletions, or other destructive actions.
- Do not attempt unauthorized security testing.
- Generate tests relevant to the user's requirements.
- Use simple CSS selectors for element_exists.
- If a requirement cannot be tested using the supported checks,
  generate an appropriate observable alternative.
- Each test case should contain at least one check.
- Return valid JSON without Markdown.
"""

    result = ask_ai(prompt)

    # Remove Markdown fences if the model adds them.
    result = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        result.strip(),
        flags=re.IGNORECASE
    )

    data = json.loads(result)

    test_cases = data.get("test_cases", [])

    if not test_cases:
        raise ValueError("The AI did not generate any test cases.")

    allowed_types = {
        "page_load",
        "title_contains",
        "url_contains",
        "text_visible",
        "element_exists"
    }

    # Validate AI-generated instructions before execution.
    validated = []

    for test in test_cases[:15]:

        if not isinstance(test, dict):
            continue

        checks = test.get("checks", [])

        if not isinstance(checks, list):
            continue

        safe_checks = []

        for check in checks:

            if not isinstance(check, dict):
                continue

            if check.get("type") not in allowed_types:
                continue

            if check["type"] in {
                "title_contains",
                "url_contains",
                "text_visible"
            }:
                if not isinstance(check.get("expected"), str):
                    continue

            if check["type"] == "element_exists":
                if not isinstance(check.get("selector"), str):
                    continue

            safe_checks.append(check)

        if safe_checks:

            validated.append({
                "name": str(test.get("name", "Unnamed test")),
                "description": str(test.get("description", "")),
                "checks": safe_checks
            })

    if not validated:
        raise ValueError("No valid test cases were generated.")

    print(f"[Agent 2] Generated {len(validated)} test cases.")

    return validated


# ============================================================
# 6. AUTOMATED TEST EXECUTION AGENT
# ============================================================

async def execute_test_cases(url, test_cases):

    print("\n[Agent 3] Executing automated browser tests...")

    results = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox"]
        )

        for index, test in enumerate(test_cases, start=1):

            print(
                f"\nRunning Test {index}: {test['name']}"
            )

            page = await browser.new_page()

            test_result = {
                "test_id": index,
                "test_name": test["name"],
                "description": test["description"],
                "status": "PASSED",
                "details": [],
                "duration_seconds": 0
            }

            start_time = datetime.now()

            try:

                response = await page.goto(
                    url,
                    wait_until="domcontentloaded",
                    timeout=45000
                )

                for check in test["checks"]:

                    check_type = check["type"]

                    try:

                        # -------------------------------
                        # Check 1: Page load
                        # -------------------------------

                        if check_type == "page_load":

                            passed = (
                                response is not None
                                and 200 <= response.status < 400
                            )

                            detail = (
                                f"HTTP status: "
                                f"{response.status if response else 'None'}"
                            )

                        # -------------------------------
                        # Check 2: Title contains
                        # -------------------------------

                        elif check_type == "title_contains":

                            actual_title = await page.title()

                            expected = check["expected"]

                            passed = (
                                expected.casefold()
                                in actual_title.casefold()
                            )

                            detail = (
                                f"Expected title to contain "
                                f"'{expected}'. "
                                f"Actual title: '{actual_title}'"
                            )

                        # -------------------------------
                        # Check 3: URL contains
                        # -------------------------------

                        elif check_type == "url_contains":

                            expected = check["expected"]

                            passed = (
                                expected.casefold()
                                in page.url.casefold()
                            )

                            detail = (
                                f"Expected URL to contain "
                                f"'{expected}'. "
                                f"Actual URL: '{page.url}'"
                            )

                        # -------------------------------
                        # Check 4: Text visible
                        # -------------------------------

                        elif check_type == "text_visible":

                            expected = check["expected"]

                            locator = page.get_by_text(
                                expected,
                                exact=False
                            )

                            passed = False

                            if await locator.count() > 0:

                                for i in range(
                                    min(await locator.count(), 10)
                                ):

                                    if await locator.nth(i).is_visible():
                                        passed = True
                                        break

                            detail = (
                                f"Text '{expected}' "
                                f"{'is' if passed else 'is not'} visible"
                            )

                        # -------------------------------
                        # Check 5: Element exists
                        # -------------------------------

                        elif check_type == "element_exists":

                            selector = check["selector"]

                            locator = page.locator(selector)

                            passed = await locator.count() > 0

                            detail = (
                                f"Selector '{selector}' "
                                f"{'exists' if passed else 'does not exist'}"
                            )

                        else:

                            passed = False
                            detail = "Unsupported check type"

                        test_result["details"].append({
                            "check": check_type,
                            "status": (
                                "PASSED" if passed else "FAILED"
                            ),
                            "detail": detail
                        })

                        if not passed:
                            test_result["status"] = "FAILED"

                    except Exception as check_error:

                        test_result["status"] = "FAILED"

                        test_result["details"].append({
                            "check": check_type,
                            "status": "ERROR",
                            "detail": str(check_error)[:500]
                        })

                # Save screenshot for failed tests.
                if test_result["status"] == "FAILED":

                    screenshot_path = (
                        f"/content/testforge_failure_{index}.png"
                    )

                    try:

                        await page.screenshot(
                            path=screenshot_path,
                            full_page=True,
                            timeout=10000
                        )

                        test_result["screenshot"] = screenshot_path

                    except Exception:
                        test_result["screenshot"] = None

            except Exception as e:

                test_result["status"] = "ERROR"

                test_result["details"].append({
                    "check": "browser_execution",
                    "status": "ERROR",
                    "detail": str(e)[:500]
                })

            finally:

                test_result["duration_seconds"] = round(
                    (datetime.now() - start_time).total_seconds(),
                    2
                )

                results.append(test_result)

                await page.close()

                print(
                    f"Result: {test_result['status']}"
                )

        await browser.close()

    print("\n[Agent 3] Test execution completed.")

    return results


# ============================================================
# 7. AI BUG ANALYSIS AGENT
# ============================================================

def analyze_failures(results, page_info):

    print("\n[Agent 4] Analyzing test failures using AI...")

    failed_results = [
        result for result in results
        if result["status"] in {"FAILED", "ERROR"}
    ]

    if not failed_results:

        return (
            "All executed tests passed. "
            "This does not guarantee that the entire website is bug-free."
        )

    prompt = f"""
You are a software debugging assistant.

Analyze these observed test failures.

WEBSITE:
{json.dumps({
    "url": page_info["final_url"],
    "title": page_info["title"]
}, indent=2)}

FAILED TEST RESULTS:
{json.dumps(failed_results, indent=2)[:12000]}

For each failure, explain:

1. Test name.
2. What was observed.
3. What the failure might indicate.
4. Possible root cause.
5. Recommended debugging step.
6. Suggested severity: Low, Medium, or High.

IMPORTANT:
- Distinguish observed facts from hypotheses.
- Do not claim that a suspected root cause is confirmed.
- A failed test does not always mean the application has a bug.
- Network problems, dynamic content, incorrect expectations, and
  browser timing can also cause failures.
- Do not invent source code that you have not inspected.

Write a clear, structured report.
"""

    try:
        return ask_ai(prompt, temperature=0.2)

    except Exception as e:
        return f"AI failure analysis could not be completed: {e}"


# ============================================================
# 8. GENERATE HTML TEST REPORT
# ============================================================

def generate_report(url, requirements, page_info, results, analysis):

    print("\n[Agent 5] Generating testing report...")

    df = pd.DataFrame(results)

    total = len(results)

    passed = sum(
        r["status"] == "PASSED" for r in results
    )

    failed = sum(
        r["status"] == "FAILED" for r in results
    )

    errors = sum(
        r["status"] == "ERROR" for r in results
    )

    pass_rate = round(
        (passed / total) * 100, 2
    ) if total else 0

    report_rows = []

    for result in results:

        details_html = "<br>".join(
            f"<b>{d['status']}</b>: {d['detail']}"
            for d in result["details"]
        )

        report_rows.append({
            "Test ID": result["test_id"],
            "Test Name": result["test_name"],
            "Status": result["status"],
            "Duration (seconds)": result["duration_seconds"],
            "Details": details_html
        })

    report_df = pd.DataFrame(report_rows)

    table_html = report_df.to_html(
        index=False,
        escape=True,
        classes="results-table",
        border=0
    )

    # Restore line breaks in test details.
    table_html = table_html.replace("&lt;br&gt;", "<br>")

    safe_analysis = (
        analysis.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )

    safe_analysis = safe_analysis.replace(
        "\n", "<br>"
    )

    safe_url = (
        url.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )

    safe_title = (
        str(page_info["title"])
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )

    safe_requirements = (
        requirements.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )

    html_report = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <title>TestForge AI Report</title>

        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 35px;
                background: #f4f6f9;
                color: #222;
            }}

            h1, h2 {{
                color: #1d3557;
            }}

            .card-container {{
                display: flex;
                flex-wrap: wrap;
                gap: 15px;
                margin: 25px 0;
            }}

            .card {{
                background: white;
                padding: 20px;
                border-radius: 10px;
                min-width: 140px;
                box-shadow: 0 2px 8px #ddd;
            }}

            .number {{
                font-size: 28px;
                font-weight: bold;
            }}

            table {{
                width: 100%;
                border-collapse: collapse;
                background: white;
            }}

            th, td {{
                padding: 12px;
                border: 1px solid #ddd;
                text-align: left;
                vertical-align: top;
                overflow-wrap: anywhere;
            }}

            th {{
                background: #1d3557;
                color: white;
            }}

            pre {{
                white-space: pre-wrap;
                overflow-wrap: anywhere;
            }}

            .analysis {{
                background: white;
                padding: 20px;
                border-radius: 10px;
                line-height: 1.7;
            }}
        </style>
    </head>

    <body>

        <h1>TestForge AI</h1>

        <p><b>Report generated:</b> {datetime.now()}</p>
        <p><b>Website:</b> {safe_url}</p>
        <p><b>Page title:</b> {safe_title}</p>
        <p><b>Requirements:</b> {safe_requirements}</p>

        <div class="card-container">

            <div class="card">
                <div>Total Tests</div>
                <div class="number">{total}</div>
            </div>

            <div class="card">
                <div>Passed</div>
                <div class="number">{passed}</div>
            </div>

            <div class="card">
                <div>Failed</div>
                <div class="number">{failed}</div>
            </div>

            <div class="card">
                <div>Execution Errors</div>
                <div class="number">{errors}</div>
            </div>

            <div class="card">
                <div>Pass Rate</div>
                <div class="number">{pass_rate}%</div>
            </div>

        </div>

        <h2>Test Execution Results</h2>

        {table_html}

        <h2>AI Failure Analysis</h2>

        <div class="analysis">
            {safe_analysis}
        </div>

        <p>
            <b>Important:</b> A passing test only confirms the
            specific checks executed. It does not establish that
            the entire website is free of defects.
        </p>

    </body>
    </html>
    """

    html_path = "/content/testforge_report.html"
    csv_path = "/content/testforge_results.csv"

    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_report)

    # Export test results and check-level details.
    flat_rows = []

    for result in results:

        for detail in result["details"]:

            flat_rows.append({
                "test_id": result["test_id"],
                "test_name": result["test_name"],
                "test_status": result["status"],
                "duration_seconds": result["duration_seconds"],
                "check_type": detail["check"],
                "check_status": detail["status"],
                "details": detail["detail"],
                "screenshot": result.get("screenshot", "")
            })

    pd.DataFrame(flat_rows).to_csv(
        csv_path,
        index=False
    )

    print("\nReport generation completed.")

    return html_path, csv_path


# ============================================================
# 9. RUN THE COMPLETE TESTFORGE AI PIPELINE
# ============================================================

async def run_testforge():

    try:

        # Agent 1: Inspect website.
        page_info = await inspect_website(target_url)

        print("\nWEBSITE INFORMATION")
        print("-" * 50)
        print("Title:", page_info["title"])
        print("Final URL:", page_info["final_url"])
        print("HTTP Status:", page_info["http_status"])

        # Agent 2: Generate test cases.
        test_cases = generate_test_cases(
            page_info,
            requirements,
            number_of_tests
        )

        print("\nGENERATED TEST CASES")
        print("-" * 50)

        for i, test in enumerate(test_cases, start=1):

            print(f"\n{i}. {test['name']}")

            for check in test["checks"]:
                print("   ", check)

        # Agent 3: Execute browser tests.
        results = await execute_test_cases(
            target_url,
            test_cases
        )

        # Agent 4: Analyze failures.
        analysis = analyze_failures(
            results,
            page_info
        )

        # Agent 5: Generate reports.
        html_path, csv_path = generate_report(
            target_url,
            requirements,
            page_info,
            results,
            analysis
        )

        # Final summary.
        df = pd.DataFrame(results)

        total = len(results)

        passed = sum(
            r["status"] == "PASSED"
            for r in results
        )

        failed = sum(
            r["status"] == "FAILED"
            for r in results
        )

        errors = sum(
            r["status"] == "ERROR"
            for r in results
        )

        pass_rate = round(
            passed / total * 100, 2
        ) if total else 0

        print("\n" + "=" * 60)
        print("             TESTFORGE AI SUMMARY")
        print("=" * 60)

        print("Total tests :", total)
        print("Passed      :", passed)
        print("Failed      :", failed)
        print("Errors      :", errors)
        print("Pass rate   :", f"{pass_rate}%")

        print("\nAI FAILURE ANALYSIS")
        print("-" * 50)
        print(analysis)

        print("\nDOWNLOAD YOUR REPORTS")
        print("-" * 50)

        display(FileLink(html_path))
        display(FileLink(csv_path))

        print("\nHTML REPORT PREVIEW")
        display(HTML(open(html_path, encoding="utf-8").read()))

        return {
            "page_info": page_info,
            "test_cases": test_cases,
            "results": results,
            "analysis": analysis,
            "html_report": html_path,
            "csv_report": csv_path
        }

    except Exception as e:

        print("\nTESTFORGE AI FAILED")
        print(type(e).__name__ + ":", str(e))

        print(
            "\nCheck your API key, model availability, "
            "website URL, network access, and browser setup."
        )

        return None


# Start the agent.
testforge_results = await run_testforge()

       TESTFORGE AI - SOFTWARE TESTING AGENT
